## 数据预处理

In [15]:
import torch
from PIL import Image
import torchvision.transforms as T
import os
from torch.utils.data import Dataset, random_split, DataLoader

In [6]:
# 自定义数据集类
class NoisyImageDataset(Dataset):
    # 初始化：传入图片根目录，以及预处理转换操作
    def __init__(self, main_dir, transform=None):
        self.main_dir = main_dir
        self.transform = transform
        self.img_names = os.listdir(main_dir)

    # 获取数据集大小
    def __len__(self):
        return len(self.img_names)

    # 根据索引号得到（input, target）
    def __getitem__(self, idx):
        # 1. 根据索引号找到文件名，读取图片数据
        img_path = os.path.join(self.main_dir, self.img_names[idx])
        img = Image.open(img_path).convert('RGB')
        # 2. 将原始图片转换为符合模型输入要求的张量,这就是重构图像的目标
        if self.transform is not None:
            img_original_tensor = self.transform(img)
        else:
            raise ValueError("Transform must be provided!")
        # 3. 添加随机噪声（高斯噪声），构建输入数据
        noise_factor = 0.5
        img_noise_tensor = img_original_tensor + torch.randn_like(img_original_tensor) * noise_factor
        img_noise_tensor = img_noise_tensor.clamp(0., 1.)
        # 将输入和目标返回
        return img_noise_tensor, img_original_tensor

In [7]:
# 定义转换操作
transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
])

In [8]:
dataset = NoisyImageDataset(main_dir='../common/dataset', transform=transform)
print(len(dataset))

24853


In [13]:
print(dataset[0])
print(dataset[1][0].shape)

(tensor([[[0.7235, 0.7372, 1.0000,  ..., 0.0653, 1.0000, 1.0000],
         [1.0000, 0.9280, 0.5996,  ..., 0.0000, 1.0000, 0.6502],
         [1.0000, 0.8318, 1.0000,  ..., 0.8862, 1.0000, 1.0000],
         ...,
         [1.0000, 1.0000, 1.0000,  ..., 0.3772, 1.0000, 1.0000],
         [0.8539, 1.0000, 0.6188,  ..., 0.6729, 1.0000, 0.9720],
         [0.5737, 0.9512, 0.5293,  ..., 0.7300, 0.6839, 0.8173]],

        [[1.0000, 0.8217, 1.0000,  ..., 0.8954, 0.7851, 1.0000],
         [0.2635, 1.0000, 0.7352,  ..., 0.5707, 1.0000, 1.0000],
         [1.0000, 0.6939, 1.0000,  ..., 1.0000, 1.0000, 0.0052],
         ...,
         [0.9608, 0.8660, 0.2641,  ..., 1.0000, 0.5516, 1.0000],
         [0.8172, 0.4083, 0.8413,  ..., 0.9420, 1.0000, 1.0000],
         [0.7519, 0.3900, 1.0000,  ..., 1.0000, 1.0000, 0.7732]],

        [[1.0000, 1.0000, 1.0000,  ..., 0.1241, 0.8380, 0.9992],
         [1.0000, 0.5860, 1.0000,  ..., 1.0000, 0.9827, 0.2229],
         [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 0

In [12]:
# 划分数据集
train_dataset, test_dataset = random_split(dataset, [0.75, 0.25])
print(len(train_dataset))
print(len(test_dataset))

18640
6213


In [16]:
# 构建数据加载器
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, drop_last=True)

In [17]:
data_iter = iter(train_loader)

In [18]:
input, target = next(data_iter)
print(input.shape)
print(target.shape)

torch.Size([32, 3, 64, 64])
torch.Size([32, 3, 64, 64])


## 定义模型

In [23]:
import torch.nn as nn
# 自定义自编码器模型
class ConvDenoiser(nn.Module):
    def __init__(self):
        super(ConvDenoiser, self).__init__()
        # 编码器部分
        # 卷积层
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 16, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        # 池化层(通用)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 解码器部分
        # 转置卷积层
        self.conv_t1 = nn.ConvTranspose2d(8, 8, kernel_size=2, stride=2)
        self.conv_t2 = nn.ConvTranspose2d(8, 16, kernel_size=2, stride=2)
        self.conv_t3 = nn.ConvTranspose2d(16, 32, kernel_size=2, stride=2)
        # 输出前普通卷积
        self.conv_out = nn.Conv2d(32, 3, kernel_size=3, padding=1)

    # 前向传播
    def forward(self, x):
        # 第一层卷积-池化
        x = torch.relu(self.conv1(x))
        # print("conv1 shape: ", x.shape)
        x = self.pool(x)
        # print("pool1 shape: ", x.shape)
        # 第二层卷积-池化
        x = torch.relu(self.conv2(x))
        # print("conv2 shape: ", x.shape)
        x = self.pool(x)
        # print("pool2 shape: ", x.shape)
        # 第三层卷积-池化
        x = torch.relu(self.conv3(x))
        # print("conv3 shape: ", x.shape)
        x = self.pool(x)
        # print("pool3 shape: ", x.shape)
        # print("Encoder output shape: ", x.shape)
        # 解码
        x = torch.relu(self.conv_t1(x))
        # print("conv1 t1 shape: ", x.shape)
        x = torch.relu(self.conv_t2(x))
        # print("conv2 t2 shape: ", x.shape)
        x = torch.relu(self.conv_t3(x))
        # print("conv3 t3 shape: ", x.shape)
        # 最后普通卷积
        x = torch.sigmoid(self.conv_out(x))
        return x

In [24]:
model = ConvDenoiser()
print(model)

ConvDenoiser(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_t1): ConvTranspose2d(8, 8, kernel_size=(2, 2), stride=(2, 2))
  (conv_t2): ConvTranspose2d(8, 16, kernel_size=(2, 2), stride=(2, 2))
  (conv_t3): ConvTranspose2d(16, 32, kernel_size=(2, 2), stride=(2, 2))
  (conv_out): Conv2d(32, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


In [25]:
output = model(input)
print(output.shape)

torch.Size([32, 3, 64, 64])


## 模型训练

In [26]:
# 1. 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ConvDenoiser(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_t1): ConvTranspose2d(8, 8, kernel_size=(2, 2), stride=(2, 2))
  (conv_t2): ConvTranspose2d(8, 16, kernel_size=(2, 2), stride=(2, 2))
  (conv_t3): ConvTranspose2d(16, 32, kernel_size=(2, 2), stride=(2, 2))
  (conv_out): Conv2d(32, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

In [27]:
# 2. 定义超参数
lr = 0.001
epochs = 10

In [28]:
import torch.optim as optim
# 3. 定义损失函数和优化器
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [29]:
# 4. 训练流程
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for input, target in train_loader:
        input = input.to(device)
        target = target.to(device)
        # 前向传播
        output = model(input)
        # 计算损失
        loss = loss_fn(output, target)
        # 反向传播
        loss.backward()
        # 更新参数
        optimizer.step()
        # 梯度清零
        optimizer.zero_grad()
        # 累加损失
        train_loss += loss.item()
    # 计算本轮平均损失
    this_train_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, train loss: {this_train_loss}")

Epoch 1, train loss: 0.035807575356804115
Epoch 2, train loss: 0.012300895995866904
Epoch 3, train loss: 0.01134046380603334
Epoch 4, train loss: 0.010893545820797022
Epoch 5, train loss: 0.010576485496180779
Epoch 6, train loss: 0.010359375888827382
Epoch 7, train loss: 0.010164431603529403
Epoch 8, train loss: 0.010022050653075435
Epoch 9, train loss: 0.009877827196922857
Epoch 10, train loss: 0.009726736942607
